# TLDR Crypto Finance Exploration

This notebook shows a simple local workflow: backfill fixture emails, inspect the DuckDB database, run a retrieval query, and generate a short risk brief.

In [ ]:
from pathlib import Path
import duckdb
import subprocess

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
db_path = project_root / 'data' / 'tldr_crypto_finance.duckdb'
fixture_dir = project_root / 'tests' / 'fixtures'

subprocess.run([
    str(project_root / '.venv' / 'bin' / 'python'),
    '-m',
    'tldr_crypto_finance.cli',
    'run-backfill',
    str(fixture_dir),
    '--source',
    'eml',
], check=True)

In [ ]:
con = duckdb.connect(str(db_path), read_only=True)
con.execute('select count(*) as messages from raw_messages').fetchall()

In [ ]:
con.execute('select newsletter_name, section_name, extracted_title, canonical_url from v_latest_articles limit 10').fetchall()

In [ ]:
from tldr_crypto_finance.retrieval.query import query_articles
from tldr_crypto_finance.retrieval.semantic import search_similar_articles
from tldr_crypto_finance.retrieval.briefs import risk_brief

query_articles(con, topic='crypto_markets', limit=5)

In [ ]:
search_similar_articles(con, 'stablecoin liquidity reserve backing', limit=3)

In [ ]:
print(risk_brief(con, 'stablecoin', topic='crypto_markets', limit=3))